# Data Augmentation for CNN Model Training
## Module 4 - Computer Vision Learning

**⚠️ Run this notebook on Google Colab with T4 GPU enabled**

This notebook handles:
1. Implement traditional data augmentations
2. Implement advanced augmentations (MixUp, CutMix, AutoAugment)
3. Train models with different augmentation strategies
4. Compare performance across augmentation techniques
5. Visualize augmentation effects

## Setup & Imports

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

if not torch.cuda.is_available():
    print("\n⚠️ WARNING: No GPU found! Please enable GPU in Colab:")
    print("   Runtime → Change runtime type → GPU (T4 recommended)")

In [ ]:
# Mount Google Drive (only for Colab)
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted")

In [ ]:
import os
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision.transforms import AutoAugment, AutoAugmentPolicy
from PIL import Image

# Metrics
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("✓ All imports successful")

In [ ]:
# Define paths
DATASET_BASE = Path('/content/drive/MyDrive/Data Science/Computer Vision/learning-assignments/datasets/flowers')
DATASET_SPLITS = DATASET_BASE / 'splits'
TRAIN_DIR = DATASET_SPLITS / 'train'
VAL_DIR = DATASET_SPLITS / 'val'
TEST_DIR = DATASET_SPLITS / 'test'

# Create output directory
OUTPUT_DIR = Path('/content/drive/MyDrive/Data Science/Computer Vision/learning-assignments/flower_classifier_augmentation_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Train directory: {TRAIN_DIR}")
print(f"Train directory exists: {TRAIN_DIR.exists()}")
print(f"Output directory: {OUTPUT_DIR}")

## Step 1: Define Data Augmentation Strategies

In [ ]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
NUM_CLASSES = 3

# Normalization parameters
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# Base transform (no augmentation)
base_transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

# Traditional augmentations only
traditional_transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),  # Translation
    transforms.RandomCrop(size=(IMG_HEIGHT, IMG_WIDTH), pad_if_needed=True, padding_mode='reflect'),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.RandomToTensor() if hasattr(transforms, 'RandomToTensor') else transforms.ToTensor(),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

# Fix the traditional transform (remove duplicate ToTensor)
traditional_transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

# AutoAugment (advanced automatic augmentation)
autoaugment_transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    AutoAugment(policy=AutoAugmentPolicy.IMAGENET),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

# Combined: Traditional + AutoAugment
combined_transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    AutoAugment(policy=AutoAugmentPolicy.IMAGENET),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

# Validation transform (no augmentation for validation)
val_transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

augmentation_strategies = {
    'no_augmentation': base_transform,
    'traditional': traditional_transform,
    'autoaugment': autoaugment_transform,
    'combined': combined_transform
}

print("✓ Augmentation strategies configured")
for name in augmentation_strategies.keys():
    print(f"  - {name}")

## Step 2: Advanced Augmentations (MixUp, CutMix)

In [ ]:
class MixUp:
    """MixUp augmentation: blend two random images and their labels."""
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def __call__(self, batch_images, batch_labels):
        batch_size = batch_images.size(0)
        indices = torch.randperm(batch_size)
        mixed_images = batch_images.clone()
        mixed_labels = batch_labels.clone().float()

        for i in range(batch_size):
            lam = np.random.beta(self.alpha, self.alpha)
            mixed_images[i] = lam * batch_images[i] + (1 - lam) * batch_images[indices[i]]
            mixed_labels[i] = lam * batch_labels[i] + (1 - lam) * batch_labels[indices[i]]

        return mixed_images, mixed_labels


class CutMix:
    """CutMix augmentation: replace rectangular region with another image."""
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def __call__(self, batch_images, batch_labels):
        batch_size = batch_images.size(0)
        _, _, height, width = batch_images.shape
        indices = torch.randperm(batch_size)
        mixed_images = batch_images.clone()
        mixed_labels = batch_labels.clone().float()

        for i in range(batch_size):
            lam = np.random.beta(self.alpha, self.alpha)

            cut_ratio = np.sqrt(1.0 - lam)
            cut_h = int(height * cut_ratio)
            cut_w = int(width * cut_ratio)

            cx = np.random.randint(0, width)
            cy = np.random.randint(0, height)

            bbx1 = np.clip(cx - cut_w // 2, 0, width)
            bby1 = np.clip(cy - cut_h // 2, 0, height)
            bbx2 = np.clip(cx + cut_w // 2, 0, width)
            bby2 = np.clip(cy + cut_h // 2, 0, height)

            mixed_images[i, :, bby1:bby2, bbx1:bbx2] = batch_images[indices[i], :, bby1:bby2, bbx1:bbx2]

            lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (height * width))
            mixed_labels[i] = lam * batch_labels[i] + (1 - lam) * batch_labels[indices[i]]

        return mixed_images, mixed_labels


class RandomErasing:
    """Random Erasing augmentation: erase rectangular regions."""
    def __init__(self, p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.0)):
        self.p = p
        self.scale = scale
        self.ratio = ratio

    def __call__(self, img):
        if np.random.random() > self.p:
            return img

        _, h, w = img.shape
        area = h * w

        for _ in range(100):
            erase_area = area * np.random.uniform(self.scale[0], self.scale[1])
            aspect_ratio = np.random.uniform(self.ratio[0], self.ratio[1])

            eh = int(np.sqrt(erase_area * aspect_ratio))
            ew = int(np.sqrt(erase_area / aspect_ratio))

            if eh < h and ew < w:
                top = np.random.randint(0, h - eh)
                left = np.random.randint(0, w - ew)
                img[:, top : top + eh, left : left + ew] = torch.randn_like(img[:, top : top + eh, left : left + ew])
                break

        return img


print("✓ Advanced augmentation classes defined (MixUp, CutMix, RandomErasing)")

## Step 3: Visualize Augmentation Effects

In [ ]:
# Load a sample image for visualization
sample_image_path = list(TRAIN_DIR.glob('*/*.jpg'))[0]
sample_image = Image.open(sample_image_path).convert('RGB')

print(f"Sample image: {sample_image_path.name}")
print(f"Original size: {sample_image.size}")

# Apply different augmentations
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Original
axes[0, 0].imshow(sample_image)
axes[0, 0].set_title('Original Image', fontweight='bold')
axes[0, 0].axis('off')

# No augmentation
aug_img = base_transform(sample_image)
aug_img_vis = aug_img.numpy().transpose((1, 2, 0))
aug_img_vis = np.array([0.229, 0.224, 0.225]) * aug_img_vis + np.array([0.485, 0.456, 0.406])
aug_img_vis = np.clip(aug_img_vis, 0, 1)
axes[0, 1].imshow(aug_img_vis)
axes[0, 1].set_title('No Augmentation', fontweight='bold')
axes[0, 1].axis('off')

# Traditional augmentation (multiple examples)
for i in range(4):
    aug_img = traditional_transform(sample_image)
    aug_img_vis = aug_img.numpy().transpose((1, 2, 0))
    aug_img_vis = np.array([0.229, 0.224, 0.225]) * aug_img_vis + np.array([0.485, 0.456, 0.406])
    aug_img_vis = np.clip(aug_img_vis, 0, 1)

    if i == 0:
        axes[0, 2].imshow(aug_img_vis)
        axes[0, 2].set_title('Traditional Augmentation', fontweight='bold')
        axes[0, 2].axis('off')
    elif i == 1:
        axes[1, 0].imshow(aug_img_vis)
        axes[1, 0].set_title('Traditional Augmentation', fontweight='bold')
        axes[1, 0].axis('off')
    elif i == 2:
        aug_img = autoaugment_transform(sample_image)
        aug_img_vis = aug_img.numpy().transpose((1, 2, 0))
        aug_img_vis = np.array([0.229, 0.224, 0.225]) * aug_img_vis + np.array([0.485, 0.456, 0.406])
        aug_img_vis = np.clip(aug_img_vis, 0, 1)
        axes[1, 1].imshow(aug_img_vis)
        axes[1, 1].set_title('AutoAugment', fontweight='bold')
        axes[1, 1].axis('off')
    else:
        aug_img = combined_transform(sample_image)
        aug_img_vis = aug_img.numpy().transpose((1, 2, 0))
        aug_img_vis = np.array([0.229, 0.224, 0.225]) * aug_img_vis + np.array([0.485, 0.456, 0.406])
        aug_img_vis = np.clip(aug_img_vis, 0, 1)
        axes[1, 2].imshow(aug_img_vis)
        axes[1, 2].set_title('Combined Augmentation', fontweight='bold')
        axes[1, 2].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'augmentation_examples.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Augmentation visualization saved")

## Step 4: Build CNN Model

In [ ]:
class FlowerCNN(nn.Module):
    def __init__(self, num_classes):
        super(FlowerCNN, self).__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.fc(x)
        return x

print("✓ FlowerCNN model class defined")

## Step 5: Training Loop

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device, use_mixup=False, use_cutmix=False):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    mixup = MixUp(alpha=1.0) if use_mixup else None
    cutmix = CutMix(alpha=1.0) if use_cutmix else None

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Apply MixUp or CutMix
        if use_mixup and mixup:
            images, labels = mixup(images, labels)
        elif use_cutmix and cutmix:
            images, labels = cutmix(images, labels)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels if not (use_mixup or use_cutmix) else labels.long())

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if not (use_mixup or use_cutmix):
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total if total > 0 else 0.0
    return epoch_loss, epoch_acc


def val_epoch(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(val_loader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


print("✓ Training and validation functions defined")

## Step 6: Train Models with Different Augmentation Strategies

In [ ]:
# Training configuration
EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15
LEARNING_RATE = 1e-3

# Dictionary to store results
all_results = {}

print("\n" + "="*70)
print("STARTING MULTI-STRATEGY TRAINING")
print("="*70)

for strategy_name, train_transform in augmentation_strategies.items():
    print(f"\n{'='*70}")
    print(f"Training: {strategy_name.upper()}")
    print(f"{'='*70}")

    # Create datasets
    train_dataset = ImageFolder(root=TRAIN_DIR, transform=train_transform)
    val_dataset = ImageFolder(root=VAL_DIR, transform=val_transform)
    test_dataset = ImageFolder(root=TEST_DIR, transform=val_transform)

    # Create loaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    # Build and initialize model
    model = FlowerCNN(NUM_CLASSES).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6
    )

    # Training loop
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    patience_counter = 0
    best_epoch = 0

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = val_epoch(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        scheduler.step(val_loss)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            patience_counter = 0
            # Save checkpoint
            checkpoint_dir = OUTPUT_DIR / 'checkpoints' / strategy_name
            checkpoint_dir.mkdir(parents=True, exist_ok=True)
            torch.save(model.state_dict(), checkpoint_dir / 'best_model.pt')
        else:
            patience_counter += 1

        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d} | TL: {train_loss:.4f} | TA: {train_acc:.4f} | "
                  f"VL: {val_loss:.4f} | VA: {val_acc:.4f}")

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

    # Test evaluation
    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()

    test_accuracy = test_correct / test_total if test_total > 0 else 0.0

    # Store results
    all_results[strategy_name] = {
        'history': history,
        'best_val_acc': best_val_acc,
        'best_epoch': best_epoch,
        'test_acc': test_accuracy,
        'model': model
    }

    print(f"\nResults for {strategy_name}:")
    print(f"  Best Val Accuracy: {best_val_acc:.4f} (Epoch {best_epoch})")
    print(f"  Test Accuracy: {test_accuracy:.4f}")

print(f"\n{'='*70}")
print("✓ TRAINING COMPLETED FOR ALL STRATEGIES")
print(f"{'='*70}")

## Step 7: Compare Results Across Strategies

In [ ]:
# Create comparison table
comparison_data = []

for strategy_name, results in all_results.items():
    comparison_data.append({
        'Strategy': strategy_name,
        'Best Val Acc': results['best_val_acc'],
        'Best Epoch': results['best_epoch'],
        'Test Accuracy': results['test_acc'],
        'Final Train Acc': results['history']['train_acc'][-1],
        'Final Val Acc': results['history']['val_acc'][-1],
    })

comparison_df = pd.DataFrame(comparison_data).sort_values('Test Accuracy', ascending=False)

print("\n" + "="*80)
print("AUGMENTATION STRATEGY COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Save comparison
comparison_df.to_csv(OUTPUT_DIR / 'strategy_comparison.csv', index=False)
print("\n✓ Comparison table saved")

## Step 8: Visualize Training History Comparison

In [ ]:
# Plot comparison of training curves
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = {'no_augmentation': '#1f77b4', 'traditional': '#ff7f0e', 'autoaugment': '#2ca02c', 'combined': '#d62728'}

# Validation accuracy comparison
for strategy_name, results in all_results.items():
    history = results['history']
    axes[0].plot(history['val_acc'], label=strategy_name.replace('_', ' ').title(),
                color=colors.get(strategy_name, None), linewidth=2.5, alpha=0.8)

axes[0].set_title('Validation Accuracy Comparison', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Validation loss comparison
for strategy_name, results in all_results.items():
    history = results['history']
    axes[1].plot(history['val_loss'], label=strategy_name.replace('_', ' ').title(),
                color=colors.get(strategy_name, None), linewidth=2.5, alpha=0.8)

axes[1].set_title('Validation Loss Comparison', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Loss', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Training comparison plots saved")

## Step 9: Bar Charts for Performance Metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

strategies = comparison_df['Strategy'].tolist()
test_accs = comparison_df['Test Accuracy'].tolist()
val_accs = comparison_df['Best Val Acc'].tolist()

x_pos = np.arange(len(strategies))
colors_list = [colors.get(s, '#7f7f7f') for s in strategies]

# Test Accuracy
bars1 = axes[0].bar(x_pos, test_accs, color=colors_list, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[0].set_ylabel('Accuracy', fontsize=11, fontweight='bold')
axes[0].set_title('Test Accuracy by Augmentation Strategy', fontsize=13, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([s.replace('_', ' ').title() for s in strategies], rotation=15, ha='right')
axes[0].set_ylim([0, 1])
axes[0].grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, acc in zip(bars1, test_accs):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')

# Best Validation Accuracy
bars2 = axes[1].bar(x_pos, val_accs, color=colors_list, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Accuracy', fontsize=11, fontweight='bold')
axes[1].set_title('Best Validation Accuracy by Strategy', fontsize=13, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([s.replace('_', ' ').title() for s in strategies], rotation=15, ha='right')
axes[1].set_ylim([0, 1])
axes[1].grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, acc in zip(bars2, val_accs):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'performance_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Performance comparison bar charts saved")

## Step 10: Detailed Evaluation of Best Model

In [ ]:
# Get best performing strategy
best_strategy = comparison_df.iloc[0]['Strategy']
best_model_results = all_results[best_strategy]
best_model = best_model_results['model']

print(f"\n{'='*70}")
print(f"BEST PERFORMING STRATEGY: {best_strategy.upper()}")
print(f"{'='*70}")
print(f"Test Accuracy: {best_model_results['test_acc']:.4f}")
print(f"Best Val Accuracy: {best_model_results['best_val_acc']:.4f}")
print(f"Best Epoch: {best_model_results['best_epoch']}")

# Load best model from checkpoint
checkpoint_dir = OUTPUT_DIR / 'checkpoints' / best_strategy
best_model.load_state_dict(torch.load(checkpoint_dir / 'best_model.pt'))
best_model.eval()

# Get class names
class_names = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])

# Evaluate on test set
test_dataset = ImageFolder(root=TEST_DIR, transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

predicted_classes = []
true_classes = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = best_model(images)
        _, predicted = torch.max(outputs, 1)
        predicted_classes.extend(predicted.cpu().numpy())
        true_classes.extend(labels.numpy())

# Classification report
print("\n" + "="*70)
print("CLASSIFICATION REPORT (TEST SET)")
print("="*70)
print(classification_report(true_classes, predicted_classes, target_names=class_names, digits=4))

# Confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'}, ax=ax)
ax.set_title(f'Confusion Matrix - {best_strategy.title()} (Test Set)', fontsize=13, fontweight='bold')
ax.set_ylabel('True Label', fontsize=11)
ax.set_xlabel('Predicted Label', fontsize=11)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix_best_model.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Best model evaluation completed")

## Step 11: Save Models & Summary Report

In [ ]:
# Save best model
model_path = OUTPUT_DIR / f'flower_classifier_best_{best_strategy}.pt'
torch.save(best_model.state_dict(), model_path)
print(f"✓ Best model saved: {model_path}")

# Create summary report
summary_report = f"""
FLOWER CLASSIFICATION WITH DATA AUGMENTATION - SUMMARY REPORT
{'='*75}

EXPERIMENT SETUP:
  Framework: PyTorch
  Model: CNN (Convolutional Neural Network)
  Input Shape: {IMG_HEIGHT}x{IMG_WIDTH}x3
  Classes: {NUM_CLASSES} ({', '.join(class_names)})
  Training Epochs: {EPOCHS}
  Batch Size: {BATCH_SIZE}
  Learning Rate: {LEARNING_RATE}
  Optimizer: Adam
  Early Stopping Patience: {EARLY_STOPPING_PATIENCE}

AUGMENTATION STRATEGIES TESTED:
  1. No Augmentation (Baseline)
     - Normalization only

  2. Traditional Augmentations
     - Horizontal Flip (50%)
     - Vertical Flip (30%)
     - Rotation (±20°)
     - Translation (±10%)
     - Color Jitter (brightness, contrast, saturation, hue)
     - Gaussian Blur (σ: 0.1-2.0)
     - Random Perspective (distortion: 0.2)

  3. AutoAugment
     - Automatic augmentation policy search (ImageNet)
     - Combines multiple augmentation operations

  4. Combined Strategy
     - Traditional augmentations + AutoAugment

ADVANCED TECHNIQUES (Available for batch processing):
  - MixUp: Blends two images and their labels
  - CutMix: Replaces rectangular regions with other images
  - Random Erasing: Randomly erases image patches

RESULTS COMPARISON:
"""

for idx, row in comparison_df.iterrows():
    summary_report += f"""
  {idx+1}. {row['Strategy'].upper()}
     - Test Accuracy: {row['Test Accuracy']:.4f}
     - Best Val Accuracy: {row['Best Val Acc']:.4f} (Epoch {int(row['Best Epoch'])})
     - Final Train Acc: {row['Final Train Acc']:.4f}
     - Improvement vs Baseline: {(row['Test Accuracy'] - comparison_df[comparison_df['Strategy']=='no_augmentation']['Test Accuracy'].values[0])*100:+.2f}%
"""

summary_report += f"""

BEST PERFORMING STRATEGY: {best_strategy.upper()}
  - Test Accuracy: {best_model_results['test_acc']:.4f}
  - Best Validation Accuracy: {best_model_results['best_val_acc']:.4f}
  - Training Epochs: {best_model_results['best_epoch']}

KEY FINDINGS:
  1. Data augmentation significantly improves generalization
  2. {best_strategy} achieved the best test performance
  3. Augmentation reduces overfitting by introducing training variability
  4. AutoAugment provides automatic policy discovery

OUTPUTS GENERATED:
  ✓ Model: flower_classifier_best_{best_strategy}.pt
  ✓ Strategy Comparison: strategy_comparison.csv
  ✓ Augmentation Examples: augmentation_examples.png
  ✓ Training Comparison: training_comparison.png
  ✓ Performance Comparison: performance_comparison.png
  ✓ Confusion Matrix: confusion_matrix_best_model.png
  ✓ Checkpoints: checkpoints/*/best_model.pt

RECOMMENDATIONS:
  1. Use augmented models for production deployment
  2. Fine-tune augmentation parameters based on your specific dataset
  3. Consider MixUp and CutMix for further improvements
  4. Monitor training/validation curves for optimal stopping point

GENERATED: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

print(summary_report)

# Save report
with open(OUTPUT_DIR / 'augmentation_summary_report.txt', 'w') as f:
    f.write(summary_report)

print(f"✓ Summary report saved")

## Step 12: Conclusions & Next Steps

In [ ]:
print("\n" + "="*75)
print("DATA AUGMENTATION TRAINING COMPLETE")
print("="*75)

print(f"""
SUMMARY:
  ✓ Trained 4 models with different augmentation strategies
  ✓ Best Strategy: {best_strategy}
  ✓ Best Test Accuracy: {best_model_results['test_acc']:.4f}
  ✓ Improvement over baseline: {(best_model_results['test_acc'] - comparison_df[comparison_df['Strategy']=='no_augmentation']['Test Accuracy'].values[0])*100:+.2f}%

OUTPUT LOCATION:
  {OUTPUT_DIR}

NEXT STEPS:
  1. Experiment with batch-level augmentations (MixUp, CutMix)
  2. Fine-tune hyperparameters for your specific use case
  3. Try transfer learning with pre-trained models
  4. Deploy the best model to production
  5. Conduct A/B testing on real data

KEY INSIGHTS:
  • Data augmentation helps prevent overfitting
  • Different strategies work differently for different datasets
  • Combining multiple augmentation techniques often yields best results
  • Augmentation is especially valuable for small datasets
""")

print("\n" + "="*75)